# Electrodynamics → Motors → Turbines → Integrated Photonics → Microfluidics
Maxwell equations → Lorentz force → rotating machines → optical waveguides → lab-on-chip.

In [ ]:
from sympy import *
from sympy.vector import CoordSys3D, gradient, divergence, curl
from IPython.display import display, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — Maxwell's Equations (Differential Form)

In [ ]:
section('Maxwell Equations')

display(Markdown(r'''
| Law | Differential form | Meaning |
|---|---|---|
| Gauss (E) | $\nabla \cdot \mathbf{E} = \rho/\varepsilon_0$ | charges source E field |
| Gauss (B) | $\nabla \cdot \mathbf{B} = 0$ | no magnetic monopoles |
| Faraday | $\nabla \times \mathbf{E} = -\partial\mathbf{B}/\partial t$ | changing B induces E |
| Ampere-Maxwell | $\nabla \times \mathbf{B} = \mu_0\mathbf{J} + \mu_0\varepsilon_0\,\partial\mathbf{E}/\partial t$ | current + changing E makes B |
'''))

# Wave equation derivation from Faraday + Ampere (vacuum)
display(Markdown(r'''
**Wave equation** from curl of Faraday:
$$\nabla \times (\nabla \times \mathbf{E}) = -\frac{\partial}{\partial t}(\nabla \times \mathbf{B})$$
Use $\nabla\times(\nabla\times\mathbf{E}) = \nabla(\nabla\cdot\mathbf{E}) - \nabla^2\mathbf{E}$ and Gauss (ρ=0):
'''))

c_sym, x, t = symbols('c x t', positive=True)
E = Function('E')(x, t)
wave_eq = Eq(E.diff(t,2), c_sym**2 * E.diff(x,2))
step('1D wave equation:', wave_eq)
sol = dsolve(wave_eq)
step('General solution (d\'Alembert):', sol)

epsilon0, mu0 = symbols('epsilon_0 mu_0', positive=True)
c_val = 1/sqrt(epsilon0*mu0)
step('Speed of light c = 1/√(ε₀μ₀) =', c_val)
step('Numerically:', '2.998 × 10⁸ m/s')

## §2 — Lorentz Force and Charged Particle Motion

In [ ]:
section('Lorentz Force')

q_charge, m_particle = symbols('q m', real=True)
E_x, E_y, E_z = symbols('E_x E_y E_z', real=True)
B_x, B_y, B_z = symbols('B_x B_y B_z', real=True)
v_x, v_y, v_z = symbols('v_x v_y v_z', real=True)

E_vec = Matrix([E_x, E_y, E_z])
B_vec = Matrix([B_x, B_y, B_z])
v_vec = Matrix([v_x, v_y, v_z])

F = q_charge * (E_vec + v_vec.cross(B_vec))
step('Lorentz force F = q(E + v×B) =', F)

# Cyclotron motion: B = B_z ẑ, E = 0
B0 = symbols('B_0', positive=True)
omega_c = q_charge * B0 / m_particle
r_c = symbols('r_c', positive=True)
step('Cyclotron frequency ω_c = qB/m =', omega_c)
step('Cyclotron radius r_c = mv⊥/(qB) =', symbols('m v_perp') / (q_charge * B0))

display(Markdown(r'''
**Circular motion in uniform B** (B = B₀ẑ):
$$x(t) = r_c \sin(\omega_c t), \quad y(t) = r_c \cos(\omega_c t)$$
This is the operating principle of **cyclotrons**, **mass spectrometers**, and plasma confinement.
'''))

## §3 — Electric Motor: Faraday's Law to Torque

In [ ]:
section('Electric Motor Physics')

N, B0, A_coil, omega_m, t_sym = symbols('N B A omega t', positive=True)
phi_angle = symbols('phi', real=True)

# Magnetic flux through rotating coil
Phi = N * B0 * A_coil * cos(omega_m * t_sym)
step('Magnetic flux Φ = NBAcos(ωt) =', Phi)

# Induced EMF (Faraday)
EMF = -diff(Phi, t_sym)
step('Induced EMF ε = −dΦ/dt =', EMF)
step('= NBAω·sin(ωt) — peak EMF =', N*B0*A_coil*omega_m)

# Torque
I_coil, R_coil = symbols('I R', positive=True)
tau_motor = N * I_coil * B0 * A_coil * sin(omega_m * t_sym)
step('Torque τ = NIBA·sin(ωt) =', tau_motor)

# Power
P_mech = tau_motor * omega_m
step('Mechanical power P = τω =', P_mech)

display(Markdown(r'''
**Motor efficiency**:
$$\eta = \frac{P_{mech}}{P_{elec}} = \frac{\tau \omega}{V I} = 1 - \frac{I^2 R_{coil}}{VI}$$
Copper loss $I^2 R$ is the dominant loss in a DC motor.

**Back-EMF constant** $K_e$: at steady state, $V_{supply} = K_e \omega + IR$
→ motor speed self-limits via back-EMF feedback.
'''))

## §4 — Turbine: Fluid Mechanics to Power

In [ ]:
section('Turbine Physics')

rho, v_wind, A_rotor, C_p = symbols('rho v A C_p', positive=True)

# Kinetic energy flux
P_wind = Rational(1,2) * rho * A_rotor * v_wind**3
step('Wind power through area A: P_wind = ½ρAv³ =', P_wind)

# Betz limit
step('Betz limit: max extractable fraction C_p ≤ 16/27 ≈ 0.593')
step('16/27 =', Rational(16,27), '=', float(Rational(16,27)))
P_turbine = C_p * P_wind
step('Turbine power P = C_p · ½ρAv³ =', P_turbine)

# Betz derivation sketch
display(Markdown(r'''
**Betz limit derivation** (momentum theory):

Let $v_1$ = upstream velocity, $v_2$ = downstream velocity, $v_d = (v_1+v_2)/2$ at disk.

Power extracted:
$$P = \frac{1}{2}\dot{m}(v_1^2 - v_2^2) = \frac{1}{2}\rho A v_d (v_1^2 - v_2^2)$$

Maximize over $v_2/v_1$: optimal at $v_2 = v_1/3 \Rightarrow C_p^{max} = 16/27$.
'''))
v1, v2 = symbols('v_1 v_2', positive=True)
vd = (v1 + v2)/2
P_extract = Rational(1,2) * rho * A_rotor * vd * (v1**2 - v2**2)
dP = diff(P_extract, v2)
v2_opt = solve(dP, v2)[0]
step('dP/dv₂ = 0 → v₂_opt =', v2_opt)
Cp_max = P_extract.subs(v2, v2_opt) / (Rational(1,2)*rho*A_rotor*v1**3)
step('C_p max =', simplify(Cp_max))

## §5 — Quantum Mechanics of EM: Photons and Cavity QED

In [ ]:
section('Quantized EM Field')

hbar, omega, n_photon = symbols('hbar omega n', positive=True)

# Photon energy
E_photon = hbar * omega
step('Photon energy E = ħω =', E_photon)

# Cavity mode energy (QHO)
E_n = hbar * omega * (n_photon + Rational(1,2))
step('Cavity mode energy E_n = ħω(n + ½) =', E_n)
step('Zero-point energy (n=0):', E_n.subs(n_photon, 0))

# Creation/annihilation
display(Markdown(r'''
**Field operators** (single mode):
$$\hat{E} \propto (\hat{a} + \hat{a}^\dagger), \quad \hat{B} \propto i(\hat{a} - \hat{a}^\dagger)$$

$$[\hat{a}, \hat{a}^\dagger] = 1$$

$$\hat{a}|n\rangle = \sqrt{n}|n-1\rangle, \quad \hat{a}^\dagger|n\rangle = \sqrt{n+1}|n+1\rangle$$

**Coherent state** $|\alpha\rangle$ (laser field): eigenstate of $\hat{a}$,
$\hat{a}|\alpha\rangle = \alpha|\alpha\rangle$, Poissonian photon statistics.
'''))

# Heisenberg uncertainty for EM field
display(Markdown(r'''
**Phase-amplitude uncertainty** (analogous to x-p):
$$\Delta n \cdot \Delta \phi \geq \frac{1}{2}$$
Laser: $\Delta n = \sqrt{\bar{n}}$ (shot noise), $\Delta\phi = 1/(2\sqrt{\bar{n}})$
→ Standard quantum limit: SNR $\leq \sqrt{\bar{n}}$
'''))

## §6 — Integrated Photonics: Waveguides and Ring Resonators

In [ ]:
section('Integrated Photonics')

n_eff, lam, R_ring, L_wg = symbols('n_eff lambda R L', positive=True)
k0 = 2*pi/lam

# Waveguide propagation constant
beta = n_eff * k0
step('Propagation constant β = n_eff · k₀ = n_eff · 2π/λ =', beta)

# Phase accumulated
phi_wg = beta * L_wg
step('Phase after length L: φ = βL =', phi_wg)

# Ring resonator condition
m = symbols('m', integer=True, positive=True)
circumference = 2*pi*R_ring
resonance = Eq(n_eff * circumference, m * lam)
step('Ring resonance condition (round-trip phase = 2πm):', resonance)
lam_res = solve(resonance, lam)[0]
step('Resonant wavelengths λ_m =', lam_res)

# FSR
FSR = lam**2 / (n_eff * circumference)
step('Free spectral range FSR = λ²/(n_eff·2πR) =', FSR)

# Q factor
Q, delta_lam = symbols('Q Delta_lambda', positive=True)
Q_expr = lam / delta_lam
step('Q factor = λ/Δλ =', Q_expr)

display(Markdown(r'''
**Silicon photonics** (SOI platform):
- Core: Si ($n=3.45$), Cladding: SiO₂ ($n=1.45$)
- Typical waveguide: 450 nm × 220 nm
- $n_{eff} \approx 2.4$ at 1550 nm
- Ring radius $R = 5\,\mu$m → FSR ≈ 20 nm
- Q ~ $10^4$–$10^6$ depending on sidewall roughness

**Applications in Jalali lab**: on-chip dispersive elements, optical time-stretch,
integrated spectrometers for single-shot phase retrieval.
'''))

## §7 — Microfluidics: Lab on a Chip

In [ ]:
section('Microfluidics')

eta, w, h, L_ch, Delta_P = symbols('eta w h L DeltaP', positive=True)
rho_fluid, v_flow, D_h = symbols('rho v D_h', positive=True)

# Reynolds number
Re = rho_fluid * v_flow * D_h / eta
step('Reynolds number Re = ρvD_h/η =', Re)
step('Microfluidics: Re ≪ 1 → laminar flow always')

# Hagen-Poiseuille: rectangular channel (approximate)
Q_flow = (w * h**3 * Delta_P) / (12 * eta * L_ch)
step('Hagen-Poiseuille flow rate Q ≈ wh³ΔP/(12ηL) =', Q_flow)

# Parabolic velocity profile
y, h_half = symbols('y h_half', real=True)
v_profile = Delta_P/(2*eta*L_ch) * (h_half**2 - y**2)
step('Parabolic velocity profile v(y) =', v_profile)
step('Max velocity at y=0:', v_profile.subs(y, 0))

# Diffusion vs convection (Peclet number)
D_diff, L_mix = symbols('D_diff L_mix', positive=True)
Pe = v_flow * L_mix / D_diff
step('Peclet number Pe = vL/D =', Pe)
step('Pe ≫ 1: convection dominates (fast flow)')
step('Pe ≪ 1: diffusion dominates (slow flow, mixing by diffusion)')

display(Markdown(r'''
**Optofluidics** (Jalali lab connection): integrate microfluidic channels
with photonic waveguides. Fluid flows through an optical cavity:
- Cells/particles scatter light → single-shot spectral signature
- TS-DFT reads the spectrum in nanoseconds per cell
- Flow cytometry at > 1 million cells/second
- Phase retrieval recovers refractive index profile of each cell
'''))

## §8 — Energy Scales: Photon to Turbine

In [ ]:
section('Energy Scales')

import numpy as np

hbar_val = 1.055e-34   # J·s
c_val    = 3e8          # m/s
lam_val  = 1550e-9      # m
omega_val = 2*np.pi*c_val/lam_val

E_photon_J = hbar_val * omega_val
E_photon_eV = E_photon_J / 1.602e-19

display(Markdown(r'''
| System | Energy | Scale |
|---|---|---|
| Single photon (1550 nm) | $\hbar\omega$ | 0.80 eV = 1.28×10⁻¹⁹ J |
| Laser pulse (1 pJ, 1 ps) | 10⁻¹² J | ~10⁷ photons |
| Electric motor (1 kW) | 10³ W × time | 10³ J/s |
| Wind turbine (2 MW) | 2×10⁶ W | 2×10⁶ J/s |
| Cyclotron proton (GeV) | 1.6×10⁻¹⁰ J | 10⁹ eV |
'''))

print(f'Photon energy at 1550 nm: {E_photon_J:.3e} J = {E_photon_eV:.3f} eV')
print(f'Photons in 1 pJ pulse:    {1e-12/E_photon_J:.2e}')
print(f'Motor at 1 kW for 1 sec:  1000 J = {1000/E_photon_J:.2e} photon-equivalents')